In [299]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler, MinMaxScaler

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

import warnings
warnings.simplefilter("ignore")

In [300]:
X_train = pd.read_csv("data/train.csv")
y_train = X_train["Survived"]
X_test = pd.read_csv("data/test.csv")
y_test = pd.read_csv("data/gender_submission.csv")["Survived"]
ids = X_test["PassengerId"]

In [301]:
X_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [302]:
# gender OneHotEncoding
ohe = OneHotEncoder(sparse_output=False, drop='first')
encoded = ohe.fit_transform(X_train[['Sex']])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
X_train = X_train.drop(columns=['Sex'])
X_train = pd.concat([X_train, encoded_df], axis=1)

encoded = ohe.transform(X_test[['Sex']])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
X_test = X_test.drop(columns=['Sex'])
X_test = pd.concat([X_test, encoded_df], axis=1)

In [303]:
# feature engineering
X_train["family"] = X_train["SibSp"] + X_train["Parch"]
X_train = X_train.drop(["SibSp", "Parch"], axis=1)

X_test["family"] = X_test["SibSp"] + X_test["Parch"]
X_test = X_test.drop(["SibSp", "Parch"], axis=1)

In [304]:
# fill Age NaN values
imputer = SimpleImputer(strategy='median')
X_train[['Age', 'Fare']] = imputer.fit_transform(X_train[['Age', 'Fare']])

X_test[['Age', 'Fare']] = imputer.transform(X_test[['Age', 'Fare']])

In [305]:
# Cabin
X_train["haveCabin"] = np.where(X_train["Cabin"].isnull(), 0, 1)
X_train = X_train.drop(["Cabin"], axis=1)

X_test["haveCabin"] = np.where(X_test["Cabin"].isnull(), 0, 1)
X_test = X_test.drop(["Cabin"], axis=1)

In [306]:
# drop
del_features = ["PassengerId", "Name", "Ticket", "Embarked"]
X_train = X_train.drop(del_features, axis=1)
X_train = X_train.drop(["Survived"], axis=1)
X_test = X_test.drop(del_features, axis=1)

In [307]:
# corr_mx = X_train.corr()
# corr_mx = X_train.corr(numeric_only=True)
# sns.heatmap(corr_mx, annot=True, fmt=".2f", cmap="coolwarm")

In [308]:
scaler = MinMaxScaler()
X_train[["Age", "Fare"]] = scaler.fit_transform(X_train[["Age", "Fare"]])
X_test[["Age", "Fare"]] = scaler.transform(X_test[["Age", "Fare"]])

In [309]:
X_train.head()

,Pclass,Age,Fare,Sex_male,family,haveCabin
0,3,0.271174,0.014151,1.0,1,0
1,1,0.472229,0.139136,0.0,1,1
2,3,0.321438,0.015469,0.0,0,0
3,1,0.434531,0.103644,0.0,1,1
4,3,0.434531,0.015713,1.0,0,0


In [310]:
# X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2)

## models

In [311]:
class Model:
    def __init__(self, model, name=None):
        self.model = model
        self.X_train, self.X_test = X_train, X_test
        self.y_train, self.y_test = y_train, y_test
        self.name = name

    def pipeline(self):
        print(f"----- {self.name if self.name else self.model.__class__.__name__} -----")
        
        
        print("Training...")
        self.model.fit(self.X_train, self.y_train)
    
        print("result:")
        pred = self.model.predict(X_test)
    
        acc = accuracy_score(self.y_test, pred)
        print(f"acc: {round(acc, 5)}")

        # print("cvs calculating...")
        # cvs = cross_val_score(self.model, self.X_train, self.y_train, scoring='accuracy', cv=5)
        # print(f"cvs: {round(cvs.mean(), 5)}")
    
        return pred

#### Random Forest

In [312]:
rf_pred = Model(RandomForestClassifier()).pipeline()

----- RandomForestClassifier -----
Training...
result:
acc: 0.82297


In [264]:
rf = RandomForestClassifier()

rf_hyperparameters = {
     'max_depth': [2, 12, 24, None],
     'n_estimators': [5, 25, 50, 100],
 }

cross_val = GridSearchCV(rf, rf_hyperparameters, cv=5)

rf_cv_pred = Model(cross_val).pipeline()

----- GridSearchCV -----
Training...
result:
acc: 0.85167
cvs: 0.84123


#### XGBoost

In [275]:
xgb_pred = Model(XGBClassifier()).pipeline()

----- XGBClassifier -----
Training...
result:
acc: 0.83971
cvs calculating...
cvs: 0.82864


In [276]:
xgb = XGBClassifier()

xgb_hyperparameters = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

cross_val = GridSearchCV(xgb, xgb_hyperparameters, cv=5, n_jobs=-1)

xgb_cv_pred = Model(cross_val).pipeline()

----- GridSearchCV -----
Training...
result:
acc: 0.89234
cvs calculating...
cvs: 0.82999


In [277]:
submission = pd.DataFrame({
    'PassengerId': ids,
    'Survived': xgb_cv_pred
})

submission.to_csv('xgb_cv_submission.csv', index=False)